In [1]:
import os 

In [2]:
os.chdir("..//")

In [3]:
os.chdir("..//")

In [4]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model'

In [5]:
import time 
import math 
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
@dataclass
class GPTConfig:
    block_size:int  = 1024  # ==> block size
    vocab_size:int  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer:int     = 12
    n_head:int      = 12
    n_emb:int       = 768   # ==> embedding dim

In [8]:
import tiktoken 

class DataLoaderLite:
    def __init__(self,B,T):
        self.B  = B     # batch 
        self.T  = T     # sequence length 
        with open("data\gpt_train.txt","r") as f:
            text    = f.read()
        enc         = tiktoken.get_encoding("gpt2")
        tokens      = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f"loaded of {len(self.tokens)} tokens")
        print(f"1 Epoch = {len(self.tokens) // (B*T)} Batches of token")

        self.current_position = 0 

    def next_batch(self):
        B,T     = self.B,self.T
        buff    = self.tokens[self.current_position:self.current_position+B*T+1]
        x       = (buff[:-1]).view(B,T)     # input 
        y       = (buff[1:]).view(B,T)      # target 
        self.current_position   += B*T 
        if self.current_position + (B*T+1)> len(self.tokens):
            self.current_position = 0 
        return x,y

In [9]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection
        self.c_proj.NANOGPT_SCALE_INIT = 1 # flaging this instance -->> Intitialization method <<--  

        self.n_head = config.n_head
        self.n_emb  = config.n_emb
        self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        ## Attention 
        #atten   = (q@k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        #atten   = atten.masked_fill(self.bias[:,:,:T,:T] == 0,float("-inf"))
        #atten   = F.softmax(atten,dim=-1)

        #y       = atten @ v                 # (B,nh,T,T) x (B,nh,T,hs) ==> (B,nh,T,hs)
        y       = F.scaled_dot_product_attention(q,k,v,is_causal=True) 
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y


In [10]:

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x
    
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x



In [11]:
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
            wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
            h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
            ln_f    = nn.LayerNorm(config.n_emb)
        ))
        self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
        # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
        #-----------------------weight sharing scheme ---------------------------------# 
        self.transformer.wte.weight  = self.lm_head.weight
        # ----------------------Parameter Initialization ------------------------------#
        self.apply(self._init_weights) 

    def _init_weights(self,module):
        std = 0.02 
        if isinstance(module,nn.Linear):
            if hasattr(module,"NANOGPT_SCALE_INIT"):
                std *= (2* self.config.n_layer) ** -0.5  # In a block there is 2 residual connection per layer, so total n_layer * 2 residual connection total 
            # (2* self.config.n_layer) ** -0.5 ==> this means 1 / sqrt(total residual connection)
            torch.nn.init.normal_(module.weight,mean = 0.0,std = std)  # weight initialization function 
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias) 
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0,std = std) 

    def forward(self,idx,target=None):
        # shape of idx is (B,T)
        B,T     = idx.shape
        assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
        pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
        pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
        tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

        x       = tok_emb + pos_emb         # (B,T,n_emb)
        for block in self.transformer.h:
            x = block(x)
        #forward the final layerorm and classifier
        x       = self.transformer.ln_f(x)
        logits  = self.lm_head(x)           # (B,T,n_emb)

        ##------------------------------Adding Target and Loss---------------------- ##
        loss    = None
        if target is None:
            loss  = None
        elif target is not None:
            loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                                    target  = target.view(-1),)
        return logits,loss


In [12]:
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

train_loader = DataLoaderLite(B=4,T=1024)
torch.set_float32_matmul_precision('high')

loaded of 338025 tokens
1 Epoch = 82 Batches of token


In [13]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

In [47]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4);

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(100):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    ## -------------------------------------Automatic mixed precision--------------------------------------------------##
    with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
        logits,loss = model(x,y)

    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()    # makes the CPU wait until the GPU finishes all its scheduled work.
    t1                  = time.time()
    dt                  = (t1 - t0) * 1000  # convert seconds to milliseconds
    token_per_second    = (train_loader.B * train_loader.T) / (t1 - t0) 
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms, tok/sec: {token_per_second:.4f}",)



c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Asus\AppData\Local\Temp\ipykernel_33836\2427792172.py:32: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  y       = F.scaled_dot_product_attention(q,k,v,is_causal=True)


Step : 0 loss: 10.9288, dt: 1835.8746 ms, tok/sec: 2231.0893
Step : 1 loss: 9.5251, dt: 2363.3235 ms, tok/sec: 1733.1525
Step : 2 loss: 8.9871, dt: 2285.3804 ms, tok/sec: 1792.2618
Step : 3 loss: 8.7009, dt: 1853.5259 ms, tok/sec: 2209.8424
Step : 4 loss: 8.4005, dt: 2282.2497 ms, tok/sec: 1794.7204
Step : 5 loss: 8.0374, dt: 1844.5525 ms, tok/sec: 2220.5928
Step : 6 loss: 7.9196, dt: 2282.7215 ms, tok/sec: 1794.3494
Step : 7 loss: 7.7211, dt: 1847.0342 ms, tok/sec: 2217.6092
Step : 8 loss: 7.6382, dt: 2282.3052 ms, tok/sec: 1794.6767
Step : 9 loss: 7.3543, dt: 1862.0925 ms, tok/sec: 2199.6759
Step : 10 loss: 7.3691, dt: 2297.1861 ms, tok/sec: 1783.0510
Step : 11 loss: 7.3623, dt: 1852.6466 ms, tok/sec: 2210.8912
Step : 12 loss: 7.4202, dt: 2296.8438 ms, tok/sec: 1783.3168
Step : 13 loss: 7.3183, dt: 1842.1810 ms, tok/sec: 2223.4515
Step : 14 loss: 6.9302, dt: 2272.3730 ms, tok/sec: 1802.5210
Step : 15 loss: 6.9384, dt: 1848.2451 ms, tok/sec: 2216.1562
Step : 16 loss: 6.7228, dt: 2275.

## ⚡ Optimizing Model Training with Powers of 2

In deep learning, small design choices can lead to significant gains in speed, memory efficiency, and hardware compatibility. One such choice is aligning your tensor shapes — especially batch sizes and embedding dimensions — to powers of 2.

### 🧠 Why Powers of 2 Matter

- Modern CPUs, GPUs, and TPUs are built to operate most efficiently on data sizes that are aligned to hardware-friendly boundaries, which are often based on powers of 2. Examples:

    - 32, 64, 128, 256, 512, 1024, etc.

- This alignment allows optimal use of:

    - Vectorized instructions (like AVX, SIMD)
    - Tensor cores
    - Cache lines
    - Memory bandwidth

### ✅ Benefits of Using Powers of 2

| Category              | Benefit                                                                 |
|-----------------------|-------------------------------------------------------------------------|
| **Hardware Optimization** | Tensor cores (NVIDIA) are optimized for shapes like 16×16 or 128×128     |
| **Memory Access**         | Aligned tensors reduce fragmentation and improve cache performance     |
| **Vectorization**         | Many CPUs/GPU cores use SIMD, which prefers power-of-2 blocks          |
| **Kernel Fusion**         | Some PyTorch/CUDA kernels fuse better with aligned shapes              |
| **Training Speed**        | Small boost (often 5–30%) in speed and stability                       |

- ### 📏 When to Use Powers of 2
    - 🟩 1. Batch Size

        - Choose batch sizes like: 16, 32, 64, 128, 256
        - This ensures optimal GPU utilization and memory alignment.
    - 🟩 2. Embedding Dimensions

        - Use 128, 256, 512, 768, 1024 as hidden sizes.
        - Transformer models especially benefit from this.
    - 🟩 3. Sequence Length (T)

        - If you’re training language models, use context lengths like 64, 128, 256, 512, 1024.
    - 🟩 4. Number of Attention Heads

            - Set number of heads such that hidden_size % n_heads == 0 and both are powers of 2.
            - Example: hidden_size = 512, n_heads = 8.
- #### 🚫 What to Avoid

    - Batch sizes like 17, 53, 113
    - → May not crash, but slower and less memory efficient.


# Change the Number from 50257 to 50304

    - 64 * 768 = 50304 

In [18]:
model = GPT(GPTConfig(vocab_size=50304))
model = model.to(device)

In [19]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(100):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    ## -------------------------------------Automatic mixed precision--------------------------------------------------##
    with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
        logits,loss = model(x,y)

    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()    # makes the CPU wait until the GPU finishes all its scheduled work.
    t1                  = time.time()
    dt                  = (t1 - t0) * 1000  # convert seconds to milliseconds
    token_per_second    = (train_loader.B * train_loader.T) / (t1 - t0) 
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms, tok/sec: {token_per_second:.4f}",)



Step : 0 loss: 10.9099, dt: 2768.1046 ms, tok/sec: 1479.7129
Step : 1 loss: 9.3669, dt: 2959.4362 ms, tok/sec: 1384.0474
Step : 2 loss: 9.0517, dt: 1836.6127 ms, tok/sec: 2230.1926
Step : 3 loss: 8.7289, dt: 2095.5358 ms, tok/sec: 1954.6314
Step : 4 loss: 8.5013, dt: 1992.5792 ms, tok/sec: 2055.6272
Step : 5 loss: 8.4996, dt: 2109.9992 ms, tok/sec: 1941.2330
Step : 6 loss: 8.3146, dt: 1986.2785 ms, tok/sec: 2062.1478
Step : 7 loss: 8.1398, dt: 2102.6864 ms, tok/sec: 1947.9843
Step : 8 loss: 7.8117, dt: 1999.0666 ms, tok/sec: 2048.9563
Step : 9 loss: 7.7495, dt: 2101.2204 ms, tok/sec: 1949.3434
Step : 10 loss: 7.5773, dt: 1987.0570 ms, tok/sec: 2061.3400
Step : 11 loss: 7.3232, dt: 2099.7074 ms, tok/sec: 1950.7480
Step : 12 loss: 7.1018, dt: 1999.2585 ms, tok/sec: 2048.7596
Step : 13 loss: 6.9657, dt: 2097.3942 ms, tok/sec: 1952.8994
Step : 14 loss: 6.8574, dt: 1986.7229 ms, tok/sec: 2061.6866
Step : 15 loss: 6.9085, dt: 2103.1508 ms, tok/sec: 1947.5541
Step : 16 loss: 6.8623, dt: 1981.

- This method is adding fake tokens, so the vocab_size has power of 2. 
- In this method, im adding extra amount of parameters into the model, and its so increase the computation of the network. 
    - 50257 * 768 = 3,85,97,376
    - 50304 * 768 = 3,86,33,472
    - extra 36096 parametres add into the network. 


## Tuning HyperParameter

In [14]:
max_lr      = 6e-4  # from GPT3 paper
min_lr      = max_lr * 0.1 #(10% of learning rate)
warmup_step = 10
max_steps   = 100
def get_learning_rate(iteration):
    # 1. linear warmup for warm_iters step 
    if iteration < warmup_step:
        return max_lr * (iteration+1) / warmup_step 
    # 2. if iteration > maximum iteration,return min learning rate 
    elif iteration > max_steps:
        return min_lr
    # 3. In between, use cosine decay down to min learning rate 
    decay_ratio = (iteration - warmup_step) / (max_steps - warmup_step)
    assert 0<= decay_ratio <=1
    coeff       = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff start at 1 and goes to 0
    return min_lr + coeff * (max_lr - min_lr)

In [15]:
model = GPT(GPTConfig(vocab_size=50304))#,block_size=2048))
model = model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=6e-4,betas=(0.9,0.95),eps=1e-8)
for iter in range(max_steps):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    ## -------------------------------------Automatic mixed precision--------------------------------------------------##
    with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
        logits,loss = model(x,y)
    loss.backward()
    norm    = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0) # for avoiding vanishing gradient
     
    #lr      = get_learning_rate(iter)
    #for params_groups in optimizer.param_groups:
        #params_groups['lr'] = lr  
    optimizer.step()
    torch.cuda.synchronize()    # makes the CPU wait until the GPU finishes all its scheduled work.
    t1                  = time.time()
    dt                  = (t1 - t0) * 1000  # convert seconds to milliseconds
    token_per_second    = (train_loader.B * train_loader.T) / (t1 - t0) 
    print(f"Step : {iter} | loss: {loss.item():.4f} | norm: {norm:.4f} | dt: {dt:.4f} ms | tok/sec: {token_per_second:.4f}tok/sec",);


Step : 0 | loss: 10.8286 | norm: 5.4359 | dt: 1101.5742 ms | tok/sec: 3718.3152
Step : 1 | loss: 8.6054 | norm: 4.0598 | dt: 987.2839 ms | tok/sec: 4148.7558
Step : 2 | loss: 8.1777 | norm: 7.7151 | dt: 1110.9865 ms | tok/sec: 3686.8136
Step : 3 | loss: 7.6365 | norm: 2.1108 | dt: 1133.4720 ms | tok/sec: 3613.6756
Step : 4 | loss: 7.2439 | norm: 2.9571 | dt: 1109.1745 ms | tok/sec: 3692.8365
Step : 5 | loss: 7.1061 | norm: 3.5914 | dt: 1131.6121 ms | tok/sec: 3619.6150
Step : 6 | loss: 6.6992 | norm: 1.8725 | dt: 1110.7833 ms | tok/sec: 3687.4878
Step : 7 | loss: 6.7428 | norm: 1.4449 | dt: 1131.0048 ms | tok/sec: 3621.5584
Step : 8 | loss: 6.9069 | norm: 1.1434 | dt: 1110.5859 ms | tok/sec: 3688.1433
Step : 9 | loss: 7.1587 | norm: 1.3634 | dt: 1131.5305 ms | tok/sec: 3619.8758
Step : 10 | loss: 7.0564 | norm: 1.1316 | dt: 1111.1467 ms | tok/sec: 3686.2820
Step : 11 | loss: 6.6908 | norm: 1.4690 | dt: 1134.1188 ms | tok/sec: 3611.6146
Step : 12 | loss: 10.0981 | norm: 18.2595 | dt: 11

In [109]:
torch.cuda.empty_cache()